In [ ]:
# ASSIGNMENT 2 – CO2
from pathlib import Path
import subprocess
import shutil
import zipfile

PROJECT = Path("/content/CSA09-192424008-Assignment2")
PROJECT.mkdir(parents=True, exist_ok=True)

if shutil.which("javac") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"],
        check=True
    )

for folder in ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q1/StringBuildBenchmark.java
import java.util.*;

public class StringBuildBenchmark {
    interface Builder { String build(String[] fields); }

    static String withString(String[] fields) {
        String result = "";
        for (String field : fields) result += field + "|";
        return result;
    }

    static String withStringBuilder(String[] fields) {
        StringBuilder result = new StringBuilder();
        for (String field : fields) result.append(field).append('|');
        return result.toString();
    }

    static String withStringBuffer(String[] fields) {
        StringBuffer result = new StringBuffer();
        for (String field : fields) result.append(field).append('|');
        return result.toString();
    }

    static long measure(Builder builder, String[] fields, int ticketCount, int repetitions) {
        long checksum = 0;
        long start = System.nanoTime();
        for (int r = 0; r < repetitions; r++)
            for (int ticket = 0; ticket < ticketCount; ticket++)
                checksum += builder.build(fields).length();
        long elapsed = System.nanoTime() - start;
        if (checksum == 0) throw new IllegalStateException("Unexpected empty result");
        return elapsed;
    }

    static String[] fields(int count) {
        String[] values = new String[count];
        for (int i = 0; i < count; i++) values[i] = "FIELD-" + i + "-DATA";
        return values;
    }

    static void runWorkload(String label, int ticketCount, int fieldCount) {
        String[] data = fields(fieldCount);
        Builder[] builders = {StringBuildBenchmark::withString,
                              StringBuildBenchmark::withStringBuilder,
                              StringBuildBenchmark::withStringBuffer};
        String[] names = {"String +", "StringBuilder", "StringBuffer"};
        int repetitions = ticketCount == 1 ? 1000 : 3;
        System.out.println("\n" + label + " | tickets=" + ticketCount +
                           " | fields/ticket=" + fieldCount + " | repeats=" + repetitions);
        for (int i = 0; i < builders.length; i++) {
            measure(builders[i], data, ticketCount, 1);
            long total = 0;
            for (int trial = 0; trial < 5; trial++)
                total += measure(builders[i], data, ticketCount, repetitions);
            double averageMs = total / 5.0 / 1_000_000.0;
            System.out.printf("%-16s average %.4f ms%n", names[i], averageMs);
        }
        System.out.println("Estimated concatenation results for String + per measured trial: " +
                           ((long) ticketCount * fieldCount * repetitions));
    }

    public static void main(String[] args) {
        Runtime runtime = Runtime.getRuntime();
        runtime.gc();
        long before = runtime.totalMemory() - runtime.freeMemory();
        runWorkload("A - Short ticket", 1, 6);
        runWorkload("B - High volume", 10000, 50);
        runWorkload("C - Long ticket", 1000, 500);
        runtime.gc();
        long after = runtime.totalMemory() - runtime.freeMemory();
        System.out.println("\nApproximate used heap before=" + before +
                           " bytes, after=" + after + " bytes");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q2/ListBenchmark.java
import java.util.*;

public class ListBenchmark {
    static long time(Runnable task) {
        long start = System.nanoTime();
        task.run();
        return System.nanoTime() - start;
    }

    static void benchmark(int n) {
        ArrayList<Integer> array = new ArrayList<>(n);
        LinkedList<Integer> linked = new LinkedList<>();
        ArrayDeque<Integer> deque = new ArrayDeque<>(n);
        for (int i = 0; i < n; i++) {
            array.add(i);
            linked.add(i);
            deque.addLast(i);
        }

        long arrayGet = time(() -> {
            long sum = 0;
            for (int i = 0; i < n; i++) sum += array.get(i);
            if (sum < 0) System.out.println(sum);
        });
        long arrayIter = time(() -> {
            long sum = 0;
            for (int value : array) sum += value;
            if (sum < 0) System.out.println(sum);
        });
        long linkedIter = time(() -> {
            long sum = 0;
            for (int value : linked) sum += value;
            if (sum < 0) System.out.println(sum);
        });
        long dequeIter = time(() -> {
            long sum = 0;
            for (int value : deque) sum += value;
            if (sum < 0) System.out.println(sum);
        });

        int operations = Math.min(1000, Math.max(100, n / 10));
        long arrayFront = time(() -> {
            ArrayList<Integer> copy = new ArrayList<>(array);
            for (int i = 0; i < operations; i++) copy.add(0, -i);
        });
        long linkedFront = time(() -> {
            LinkedList<Integer> copy = new LinkedList<>(linked);
            for (int i = 0; i < operations; i++) copy.addFirst(-i);
        });
        long dequeFront = time(() -> {
            ArrayDeque<Integer> copy = new ArrayDeque<>(deque);
            for (int i = 0; i < operations; i++) copy.addFirst(-i);
        });

        System.out.printf("%nN=%d, front operations=%d%n", n, operations);
        System.out.printf("ArrayList get-all %.3f ms; iteration %.3f ms; front insert %.3f ms%n",
                arrayGet / 1e6, arrayIter / 1e6, arrayFront / 1e6);
        System.out.printf("LinkedList iteration %.3f ms; front insert %.3f ms%n",
                linkedIter / 1e6, linkedFront / 1e6);
        System.out.printf("ArrayDeque iteration %.3f ms; front insert %.3f ms%n",
                dequeIter / 1e6, dequeFront / 1e6);
    }

    public static void main(String[] args) {
        for (int size : new int[]{1000, 10000, 100000}) benchmark(size);
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q3/Attendance.java
import java.util.*;

class Student {
    private final String regNo;
    private final String name;

    Student(String regNo, String name) {
        this.regNo = Objects.requireNonNull(regNo);
        this.name = name;
    }

    @Override
    public boolean equals(Object object) {
        if (this == object) return true;
        if (!(object instanceof Student)) return false;
        Student other = (Student) object;
        return regNo.equals(other.regNo);
    }

    @Override
    public int hashCode() {
        return Objects.hash(regNo);
    }

    @Override
    public String toString() {
        return regNo + " " + name;
    }
}

public class Attendance {
    public static void main(String[] args) {
        Set<Student> present = new HashSet<>();
        present.add(new Student("192311001", "Arun"));
        present.add(new Student("192311002", "Divya"));
        present.add(new Student("192311001", "Arun Duplicate"));

        System.out.println("Unique students: " + present.size());
        System.out.println("Contains Arun: " +
                present.contains(new Student("192311001", "Different display name")));

        Set<Student> sorted = new TreeSet<>(Comparator.comparing(s -> s.toString().split(" ")[0]));
        sorted.addAll(present);
        System.out.println("Sorted attendance: " + sorted);

        Map<String, Student> byRegisterNumber = new HashMap<>();
        for (Student student : present) {
            String key = student.toString().split(" ")[0];
            byRegisterNumber.putIfAbsent(key, student);
        }
        System.out.println("HashMap unique count: " + byRegisterNumber.size());
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q4/PharmacyMaps.java
import java.util.*;

class Medicine {
    final String code;
    final String name;
    int quantity;

    Medicine(String code, String name, int quantity) {
        this.code = code;
        this.name = name;
        this.quantity = quantity;
    }

    @Override
    public String toString() {
        return code + " " + name + " qty=" + quantity;
    }
}

public class PharmacyMaps {
    static long lookupTime(Map<String, Medicine> map, List<String> keys, int repeats) {
        long start = System.nanoTime();
        int found = 0;
        for (int r = 0; r < repeats; r++)
            for (String key : keys)
                if (map.get(key) != null) found++;
        long elapsed = System.nanoTime() - start;
        if (found < 0) System.out.println(found);
        return elapsed;
    }

    public static void main(String[] args) {
        Map<String, Medicine> hash = new HashMap<>();
        Map<String, Medicine> sorted = new TreeMap<>();
        Map<String, Medicine> insertion = new LinkedHashMap<>();

        String[][] sample = {
            {"M30", "Paracetamol", "100"},
            {"M10", "Amoxicillin", "60"},
            {"M20", "Cetirizine", "80"}
        };
        for (String[] row : sample) {
            Medicine medicine = new Medicine(row[0], row[1], Integer.parseInt(row[2]));
            hash.put(medicine.code, medicine);
            sorted.put(medicine.code, medicine);
            insertion.put(medicine.code, medicine);
        }

        System.out.println("Lookup M20: " + hash.get("M20"));
        System.out.println("Code order: " + sorted.values());
        System.out.println("Insertion order: " + insertion.values());

        for (int size : new int[]{1000, 10000, 100000}) {
            Map<String, Medicine> map = new HashMap<>(size * 2);
            List<String> successful = new ArrayList<>(size);
            List<String> unsuccessful = new ArrayList<>(size);
            for (int i = 0; i < size; i++) {
                String code = String.format("MED%07d", i);
                map.put(code, new Medicine(code, "Medicine" + i, i % 500));
                successful.add(code);
                unsuccessful.add("MISS" + i);
            }
            int repeats = Math.max(1, 100000 / size);
            for (int warm = 0; warm < 2; warm++) {
                lookupTime(map, successful, repeats);
                lookupTime(map, unsuccessful, repeats);
            }
            long successNs = lookupTime(map, successful, repeats);
            long missNs = lookupTime(map, unsuccessful, repeats);
            System.out.printf("N=%d successful %.3f ms; unsuccessful %.3f ms%n",
                    size, successNs / 1e6, missNs / 1e6);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q5/LeaderboardDemo.java
import java.util.*;

class Team {
    final String name;
    final int points;
    final int goalDiff;

    Team(String name, int points, int goalDiff) {
        this.name = name;
        this.points = points;
        this.goalDiff = goalDiff;
    }

    @Override
    public String toString() {
        return name + "(" + points + "," + goalDiff + ")";
    }
}

class Leaderboard<T> {
    private final List<T> items = new ArrayList<>();
    private final Comparator<? super T> order;

    Leaderboard(Comparator<? super T> order) {
        this.order = order;
    }

    void add(T item) { items.add(item); }

    void addAll(List<? extends T> values) { items.addAll(values); }

    List<T> ranking() {
        List<T> copy = new ArrayList<>(items);
        copy.sort(order);
        return copy;
    }
}

public class LeaderboardDemo {
    public static void main(String[] args) {
        Comparator<Team> byRank =
            Comparator.comparingInt((Team team) -> team.points).reversed()
                .thenComparing(Comparator.comparingInt((Team team) -> team.goalDiff).reversed())
                .thenComparing(team -> team.name);

        Leaderboard<Team> board = new Leaderboard<>(byRank);
        board.add(new Team("Panthers", 9, 4));
        board.add(new Team("Eagles", 9, 6));
        board.add(new Team("Cobras", 6, 1));
        board.add(new Team("Bisons", 9, 4));

        System.out.println(board.ranking());
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q6/Triage.java
import java.util.*;

class Patient {
    final String name;
    final int severity;
    final long arrival;

    Patient(String name, int severity, long arrival) {
        this.name = name;
        this.severity = severity;
        this.arrival = arrival;
    }

    @Override
    public String toString() {
        return name + "(S" + severity + ")";
    }
}

public class Triage {
    private long counter = 0;
    private final PriorityQueue<Patient> queue =
        new PriorityQueue<>(Comparator.comparingInt((Patient p) -> p.severity)
                                      .thenComparingLong(p -> p.arrival));

    void admit(String name, int severity) {
        if (severity < 1 || severity > 5)
            throw new IllegalArgumentException("Severity must be 1..5");
        queue.add(new Patient(name, severity, counter++));
    }

    Patient next() { return queue.poll(); }

    public static void main(String[] args) {
        Triage er = new Triage();
        er.admit("Ravi", 3);
        er.admit("Meena", 1);
        er.admit("John", 3);
        er.admit("Asha", 5);
        er.admit("Kumar", 1);
        er.admit("Lata", 2);
        er.admit("Nila", 4);
        er.admit("Dev", 2);

        StringBuilder order = new StringBuilder();
        Patient patient;
        while ((patient = er.next()) != null) order.append(patient).append(' ');
        System.out.println(order.toString().trim());
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment2/Q7/TicketRedesign.java
import java.util.*;

public class TicketRedesign {
    static String oldBuild(String[] fields) {
        String result = "";
        for (String field : fields) result += field + "\n";
        return result;
    }

    static String newBuild(String[] fields) {
        StringBuilder result = new StringBuilder();
        for (String field : fields) result.append(field).append('\n');
        return result.toString();
    }

    static long measure(boolean redesigned, String[] fields, int repetitions) {
        long checksum = 0;
        long start = System.nanoTime();
        for (int i = 0; i < repetitions; i++) {
            String result = redesigned ? newBuild(fields) : oldBuild(fields);
            checksum += result.length();
        }
        long elapsed = System.nanoTime() - start;
        if (checksum == 0) throw new IllegalStateException("Unexpected empty result");
        return elapsed;
    }

    public static void main(String[] args) {
        int[] sizes = {10, 50, 500};
        for (int size : sizes) {
            String[] fields = new String[size];
            for (int i = 0; i < size; i++) fields[i] = "Field-" + i + "-value";

            String oldText = oldBuild(fields);
            String newText = newBuild(fields);
            System.out.println("n=" + size + " identical=" + oldText.equals(newText));

            int repetitions = Math.max(100, 100000 / size);
            measure(false, fields, 10);
            measure(true, fields, 10);

            long oldTotal = 0;
            long newTotal = 0;
            for (int trial = 0; trial < 5; trial++) {
                oldTotal += measure(false, fields, repetitions);
                newTotal += measure(true, fields, repetitions);
            }

            double oldAverage = oldTotal / 5.0 / 1e6;
            double newAverage = newTotal / 5.0 / 1e6;
            System.out.printf("Old %.3f ms; new %.3f ms; speedup %.2fx%n",
                    oldAverage, newAverage, oldAverage / Math.max(0.000001, newAverage));
        }
    }
}


In [ ]:
TASK_FOLDERS = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]

for folder in TASK_FOLDERS:
    java_files = sorted(str(path) for path in (PROJECT / folder).glob("*.java"))
    result = subprocess.run(
        ["javac"] + java_files,
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(folder, "COMPILED" if result.returncode == 0 else "COMPILATION ERROR")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"Compilation failed in {folder}")


In [ ]:
def run_java(folder, class_name):
    result = subprocess.run(
        ["java", "-cp", str(PROJECT / folder), class_name],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"{class_name} failed")


In [ ]:
run_java("Q1", "StringBuildBenchmark")

In [ ]:
run_java("Q2", "ListBenchmark")

In [ ]:
run_java("Q3", "Attendance")

In [ ]:
run_java("Q4", "PharmacyMaps")

In [ ]:
run_java("Q5", "LeaderboardDemo")

In [ ]:
run_java("Q6", "Triage")

In [ ]:
run_java("Q7", "TicketRedesign")

In [ ]:
ZIP_PATH = Path("/content/CSA09-192424008-Assignment2.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob("*"):
        if path.is_file() and path.suffix != ".class":
            archive.write(path, path.relative_to(PROJECT.parent))
print(ZIP_PATH)
